# Phase 0: cohort and frozen data contract



In [1]:
%run definitions.ipynb
require_pythonhashseed()
set_all_seeds(SPLIT_SEED)
ENV = environment_record()
print(json.dumps(ENV, indent=2))


required packages already match the pins.
torch 2.8.0+cu128 present (training phases need >= 2.8.0).
torchvision 0.23.0+cu128 present (training phases need >= 0.23.0).


python 3.12.3 | numpy 2.2.6 | pandas 2.3.3 | pillow 12.0.0 | opencv 4.12.0 | torch available: True


{
  "timestamp_utc": "2026-08-23T20:59:32Z",
  "python": "3.12.3",
  "platform": "Linux-6.8.0-124-generic-x86_64-with-glibc2.39",
  "pythonhashseed": "2026",
  "packages": {
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "pillow": "12.0.0",
    "opencv-python-headless": "4.12.0.88",
    "scikit-learn": "1.7.2",
    "matplotlib": "3.10.7",
    "nbformat": "5.10.4",
    "torch": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128"
  },
  "definitions_sha256": "cfcf3a434b8e82ce21ef86fdf1f548d338cab1ee53a6fd1d81d91bdf8d89a813",
  "gpu": "NVIDIA A100-SXM4-80GB"
}


In [2]:
# Stage A: verify inputs and merge the label tables.
table = load_label_tables()

image_files = sorted(p.name for p in IMAGES_DIR.iterdir() if p.is_file() and not p.name.startswith("."))
if len(image_files) != EXPECTED_IMAGE_FILES:
    raise PhaseStop(f"images/ holds {len(image_files)} files, expected {EXPECTED_IMAGE_FILES}. "
                    "Re-check the folder before continuing (see restored_files.csv).")

stem_to_file = {}
for name in image_files:
    stem = os.path.splitext(name)[0]
    if stem in stem_to_file:
        raise PhaseStop(f"two files share the stem {stem!r}: {stem_to_file[stem]} and {name}.")
    stem_to_file[stem] = name

unmatched = [i for i in table["ID"] if i not in stem_to_file]
if unmatched:
    raise PhaseStop(f"{len(unmatched)} table ids have no image file, first: {unmatched[:5]}")

print(f"tables merged: {len(table)} rows | image files: {len(image_files)} | all ids resolve to one file")


tables merged: 2208 rows | image files: 2451 | all ids resolve to one file


In [3]:
# Stage B: apply the registered cohort rule and run the row-level data tests.
mask = table[TARGETS].sum(axis=1) > 0
cohort = table.loc[mask].copy().reset_index(drop=True)
if len(cohort) != EXPECTED_COHORT:
    raise PhaseStop(f"cohort rule produced {len(cohort)} rows, the registered count is {EXPECTED_COHORT}. "
                    "Stop and resolve the source version before splitting.")

cohort.rename(columns={"ID": "image_id"}, inplace=True)
cohort["relative_path"] = ["images/" + stem_to_file[i] for i in cohort["image_id"]]
cohort["source_dataset"] = [image_source(i) for i in cohort["image_id"]]
cohort["source_subject_id"] = [source_subject(i) for i in cohort["image_id"]]

# Data test 6: NORMAL together with any other positive label goes to source review.
others = [c for c in ALL_LABELS if c != "NORMAL"]
conflicts = cohort[(cohort["NORMAL"] == 1) & (cohort[others].sum(axis=1) > 0)]
conflicts[["image_id"] + ALL_LABELS].to_csv(PHASE0_REPORTS / "normal_conflicts.csv", index=False)
print(f"cohort rows: {len(cohort)} | NORMAL-plus-disease rows listed for review: {len(conflicts)}")
print(cohort["source_dataset"].value_counts().to_dict())
print({t: int(cohort[t].sum()) for t in TARGETS})


cohort rows: 1489 | NORMAL-plus-disease rows listed for review: 3
{'rfmid': 1230, 'stare': 150, 'aria': 109}
{'DR': 495, 'NORMAL': 493, 'ODC': 263, 'MH': 169, 'DN': 162}


In [4]:
# Stage C: one pass over the cohort images.
# Per image: SHA-256 of the file bytes, field-of-view scan, 224 crop written to
# the image cache, perceptual hash, and saved example crops per source.
# A verified manifest from an earlier pass skips the recompute.
CACHE_DIR = PHASE0_ARTIFACTS / "image_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXAMPLES_DIR = PHASE0_REPORTS / "preproc_examples"
EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = PHASE0_ARTIFACTS / "cache_manifest.csv"
AUDIT_PATH = PHASE0_REPORTS / "crop_audit.csv"

cohort_ids = sorted(cohort["image_id"])
path_of = dict(zip(cohort["image_id"], cohort["relative_path"]))


def stage_c_fast_path():
    if not (MANIFEST_PATH.exists() and AUDIT_PATH.exists()):
        return None
    m = pd.read_csv(MANIFEST_PATH, dtype={"image_id": str})
    need = {"image_id", "cache_file", "cache_sha256", "original_sha256", "phash",
            "pillow", "preprocessing"}
    if not need.issubset(m.columns) or sorted(m["image_id"]) != cohort_ids:
        return None
    if not ((m["pillow"] == _ilm.version("pillow")).all()
            and (m["preprocessing"] == PREPROCESSING_VERSION).all()):
        return None
    audit = pd.read_csv(AUDIT_PATH, dtype={"image_id": str})
    if sorted(audit["image_id"]) != cohort_ids or not bool(audit["ok"].all()):
        return None
    for f in m["cache_file"]:
        if not (CACHE_DIR / f).exists():
            return None
    lookup = m.set_index("image_id")
    rng = np.random.default_rng(SPLIT_SEED)
    for i in rng.choice(np.array(cohort_ids), size=20, replace=False):
        if sha256_file(PROJECT_ROOT / path_of[i]) != lookup.loc[i, "original_sha256"]:
            return None
        if sha256_file(CACHE_DIR / lookup.loc[i, "cache_file"]) != lookup.loc[i, "cache_sha256"]:
            return None
    return lookup


_fast = stage_c_fast_path()
if _fast is not None:
    sha_by_id = _fast["original_sha256"].to_dict()
    phash_by_id = {i: int(v) for i, v in _fast["phash"].to_dict().items()}
    print("stage C fast path: manifest verified with a seeded 20-image spot check")
else:
    crop_rows = []
    sha_by_id = {}
    phash_by_id = {}
    cache_rows = []
    examples_saved = {s: 0 for s in SOURCES}
    t_start = time.time()
    for j, row in enumerate(cohort.itertuples()):
        path = PROJECT_ROOT / row.relative_path
        data = path.read_bytes()
        sha_by_id[row.image_id] = sha256_bytes(data)
        arr = decode_rgb(data)
        scan = fov_scan(arr)
        crop_rows.append({"image_id": row.image_id, **scan})
        if scan["ok"]:
            crop = fov_crop_224(arr, scan)
            phash_by_id[row.image_id] = phash63(crop)
            cache_path = CACHE_DIR / f"{row.image_id}.png"
            crop.save(cache_path, format="PNG")
            cache_rows.append({"image_id": row.image_id, "cache_file": cache_path.name,
                               "cache_sha256": sha256_file(cache_path),
                               "original_sha256": sha_by_id[row.image_id],
                               "phash": phash_by_id[row.image_id]})
            if examples_saved[row.source_dataset] < 3:
                crop.save(EXAMPLES_DIR / f"{row.source_dataset}_{examples_saved[row.source_dataset]}.png")
                examples_saved[row.source_dataset] += 1
        if (j + 1) % 200 == 0:
            print(f"  {j + 1}/{len(cohort)} images, {time.time() - t_start:.0f}s")

    crop_audit = pd.DataFrame(crop_rows)
    crop_audit.to_csv(AUDIT_PATH, index=False)
    failures = crop_audit[~crop_audit["ok"]]
    if len(failures):
        failures.to_csv(PHASE0_REPORTS / "crop_failures.csv", index=False)
        raise PhaseStop(f"{len(failures)} images failed the field-of-view scan. "
                        "Review phase0/reports/crop_failures.csv; no silent fallback exists.")
    pd.DataFrame(cache_rows).assign(pillow=_ilm.version("pillow"),
                                    preprocessing=PREPROCESSING_VERSION).to_csv(MANIFEST_PATH, index=False)
    print(f"crop audit passed for {len(cohort)} images in {time.time() - t_start:.0f}s")

cohort["sha256"] = [sha_by_id[i] for i in cohort["image_id"]]
dupe_sha = cohort[cohort.duplicated("sha256", keep=False)]
if len(dupe_sha):
    dupe_sha.to_csv(PHASE0_REPORTS / "sha_duplicates.csv", index=False)
    raise PhaseStop(f"{len(dupe_sha)} cohort rows share a SHA-256. The build stops.")
print("byte-level duplicate check passed")


stage C fast path: manifest verified with a seeded 20-image spot check
byte-level duplicate check passed


In [5]:
# Stage D: duplicate audit. Hash screen, then geometric verification.
ids_sorted = sorted(phash_by_id)
hv = np.array([phash_by_id[i] for i in ids_sorted], dtype=np.uint64)
xor = hv.reshape(-1, 1) ^ hv.reshape(1, -1)
dist = np.bitwise_count(xor)
iu = np.triu_indices(len(ids_sorted), k=1)
sel = np.nonzero(dist[iu] <= PHASH_HAMMING_MAX)[0]
cand = pd.DataFrame([{"image_id_a": ids_sorted[iu[0][s]],
                      "image_id_b": ids_sorted[iu[1][s]],
                      "hamming": int(dist[iu][s])} for s in sel])
cand = cand.sort_values(["image_id_a", "image_id_b"]).reset_index(drop=True)
print(f"candidate pairs at hamming <= {PHASH_HAMMING_MAX}: {len(cand)}")

verifier = HiresVerifier(stem_to_file)
t0 = time.time()
scores = []
for k, r in enumerate(cand.itertuples()):
    scores.append(verifier.inliers(r.image_id_a, r.image_id_b))
    if (k + 1) % 1000 == 0:
        print(f"  verified {k + 1}/{len(cand)}, {time.time() - t0:.0f}s")
cand["hires_inliers"] = scores
cand["rule"] = [duplicate_rule(s) for s in scores]
cand.to_csv(PHASE0_SOURCE / "duplicate_candidates.csv", index=False)
print(cand["rule"].value_counts().to_string())

decisions_path = PHASE0_SOURCE / "duplicate_decisions.csv"
human = {}
if decisions_path.exists():
    prior = pd.read_csv(decisions_path, dtype={"image_id_a": str, "image_id_b": str}).fillna("")
    for r in prior.itertuples():
        if str(r.decision_source) == "human" and str(r.decision) in ("confirm", "reject"):
            human[tuple(sorted([r.image_id_a, r.image_id_b]))] = (str(r.decision), str(r.note))

final_rows, missing = [], []
for r in cand.itertuples():
    pair_key = tuple(sorted([r.image_id_a, r.image_id_b]))
    if r.rule == "review":
        if pair_key in human:
            decision, source, note = human[pair_key][0], "human", human[pair_key][1]
        else:
            decision, source, note = "", "human", ""
            missing.append(pair_key)
    else:
        decision, source, note = r.rule, "rule", ""
    final_rows.append({"image_id_a": pair_key[0], "image_id_b": pair_key[1],
                       "hamming": r.hamming, "hires_inliers": r.hires_inliers,
                       "decision": decision, "decision_source": source, "note": note})
decisions = pd.DataFrame(final_rows)
decisions.to_csv(decisions_path, index=False)
if missing:
    raise PhaseStop(f"{len(missing)} review-band pairs need a human decision in "
                    f"{decisions_path}: {missing[:10]}")

confirmed_pairs = sorted((r.image_id_a, r.image_id_b)
                         for r in decisions.itertuples() if r.decision == "confirm")
n_human = int((decisions["decision_source"] == "human").sum())
print(f"decisions: {len(decisions)} | confirmed: {len(confirmed_pairs)} | human rows: {n_human}")


candidate pairs at hamming <= 8: 8520


  verified 1000/8520, 47s


  verified 2000/8520, 69s


  verified 3000/8520, 90s


  verified 4000/8520, 112s


  verified 5000/8520, 133s


  verified 6000/8520, 175s


  verified 7000/8520, 194s


  verified 8000/8520, 213s


rule
reject     8511
confirm       8
review        1
decisions: 8520 | confirmed: 9 | human rows: 1


In [6]:
# Stage E: analysis groups.
# Base key: source plus subject when a subject exists, otherwise the image id.
# Confirmed near-duplicate links join their base groups. The group id is the
# SHA-256 of the sorted base keys in the connected component, first 16 hex chars.
all_ids = sorted(cohort["image_id"])
dsu = DisjointSets(all_ids)
key_first_id = {}
for i in all_ids:
    k = base_group_key(i)
    if k in key_first_id:
        dsu.union(key_first_id[k], i)
    else:
        key_first_id[k] = i
for a, b in confirmed_pairs:
    dsu.union(a, b)

components = {}
for i in all_ids:
    components.setdefault(dsu.find(i), []).append(i)

group_of = {}
cluster_of = {}
for members in components.values():
    keys = sorted({base_group_key(i) for i in members})
    gid = hashlib.sha256("|".join(keys).encode()).hexdigest()[:16]
    for i in members:
        group_of[i] = gid

for a, b in confirmed_pairs:
    root = dsu.find(a)
    members = sorted(components[root])
    cid = hashlib.sha256("|".join(members).encode()).hexdigest()[:12]
    for i in members:
        cluster_of[i] = cid

cohort["analysis_group_id"] = [group_of[i] for i in cohort["image_id"]]
cohort["duplicate_cluster_id"] = [cluster_of.get(i, "") for i in cohort["image_id"]]

sizes = cohort.groupby("analysis_group_id").size()
print(f"analysis groups: {len(sizes)} | largest group: {int(sizes.max())} images | "
      f"multi-image groups: {int((sizes > 1).sum())} | duplicate clusters: {len(set(cluster_of.values()))}")


analysis groups: 1432 | largest group: 3 images | multi-image groups: 53 | duplicate clusters: 8


In [7]:
# Write cohort.csv with the registered field order.
non_targets = [c for c in ALL_LABELS if c not in TARGETS]
cohort_cols = (["image_id", "relative_path", "source_dataset", "source_subject_id",
                "duplicate_cluster_id", "analysis_group_id"] + TARGETS + non_targets + ["sha256"])
cohort_out = cohort[cohort_cols].sort_values("image_id").reset_index(drop=True)
cohort_out.to_csv(PHASE0_ARTIFACTS / "cohort.csv", index=False)
COHORT_SHA = sha256_file(PHASE0_ARTIFACTS / "cohort.csv")
print(f"cohort.csv written, sha256 {COHORT_SHA}")


cohort.csv written, sha256 9252929f5b55e091a27e22ab18a198c2ce7705661acb7dd57afa0136f7e76e00


In [8]:
# Stage F: outer folds and inner-validation splits.
rng_outer = np.random.default_rng(SPLIT_SEED)
units = build_group_units(cohort)
capacities = [OUTER_TEST_CAPACITY[f] for f in range(1, N_FOLDS + 1)]
outer_assignment = grouped_stratified_partition(units, capacities, rng_outer)
cohort["test_fold"] = [outer_assignment[g] + 1 for g in cohort["analysis_group_id"]]

for f in range(1, N_FOLDS + 1):
    test = cohort[cohort["test_fold"] == f]
    if len(test) != OUTER_TEST_CAPACITY[f]:
        raise PhaseStop(f"fold {f} test size {len(test)} differs from {OUTER_TEST_CAPACITY[f]}.")
    check_label_support(test, f"outer test fold {f}")

roles = {}
for f in range(1, N_FOLDS + 1):
    pool = cohort[cohort["test_fold"] != f]
    val_size = len(pool) - FIT_IMAGES_PER_FOLD
    rng_inner = np.random.default_rng([SPLIT_SEED, f])
    inner_units = build_group_units(pool)
    inner_assignment = grouped_stratified_partition(inner_units, [FIT_IMAGES_PER_FOLD, val_size], rng_inner)
    for row in pool.itertuples():
        roles[(f, row.image_id)] = "fit" if inner_assignment[row.analysis_group_id] == 0 else "inner_val"
    fit = pool[[roles[(f, i)] == "fit" for i in pool["image_id"]]]
    val = pool[[roles[(f, i)] == "inner_val" for i in pool["image_id"]]]
    if len(fit) != FIT_IMAGES_PER_FOLD:
        raise PhaseStop(f"fold {f} fit pool is {len(fit)}, expected {FIT_IMAGES_PER_FOLD}.")
    check_label_support(val, f"fold {f} inner validation")
    check_label_support(fit, f"fold {f} fit pool")
    print(f"fold {f}: test {len(test) if f == N_FOLDS else OUTER_TEST_CAPACITY[f]}, "
          f"fit {len(fit)}, inner_val {len(val)}")

cohort[["image_id", "test_fold"]].sort_values("image_id").to_csv(
    PHASE0_ARTIFACTS / "outer_folds.csv", index=False)
inner_rows = [{"test_fold": f, "image_id": i, "role": r} for (f, i), r in sorted(roles.items())]
pd.DataFrame(inner_rows).to_csv(PHASE0_ARTIFACTS / "inner_splits.csv", index=False)
print("outer_folds.csv and inner_splits.csv written")


fold 1: test 298, fit 1044, inner_val 147


fold 2: test 298, fit 1044, inner_val 147


fold 3: test 298, fit 1044, inner_val 147


fold 4: test 298, fit 1044, inner_val 147


fold 5: test 297, fit 1044, inner_val 148
outer_folds.csv and inner_splits.csv written


In [9]:
# Stage G: nested training subsets.
SUBSETS_DIR = PHASE0_ARTIFACTS / "subsets"
SUBSETS_DIR.mkdir(parents=True, exist_ok=True)
support_rows = []
inner_frame = pd.DataFrame(inner_rows)

for f in range(1, N_FOLDS + 1):
    fit_ids = set(inner_frame[(inner_frame["test_fold"] == f) & (inner_frame["role"] == "fit")]["image_id"])
    fit_frame = cohort[cohort["image_id"].isin(fit_ids)][["image_id"] + TARGETS].reset_index(drop=True)
    for seed in TRAIN_SEEDS:
        subsets = nested_subsets(fit_frame, f, seed)
        previous = None
        for n in sorted(subsets):
            ids = subsets[n]
            if previous is not None and not set(previous).issubset(ids):
                raise PhaseStop(f"fold {f} seed {seed}: subset {n} does not contain the smaller subset.")
            previous = ids
            pd.DataFrame({"image_id": ids}).to_csv(
                SUBSETS_DIR / f"subset_f{f}_s{seed}_n{n}.csv", index=False)
            sub = cohort[cohort["image_id"].isin(ids)]
            support_rows.append({"fold": f, "seed": seed, "n": n,
                                 **{t: int(sub[t].sum()) for t in TARGETS}})

support = pd.DataFrame(support_rows)
support.to_csv(PHASE0_REPORTS / "subset_class_support.csv", index=False)
print(f"subset files written: {len(support)} | all nesting checks passed")
support[support["n"] == 50]


subset files written: 40 | all nesting checks passed


,fold,seed,n,DR,NORMAL,ODC,MH,DN
0,1,42,50,17,18,9,6,5
5,1,43,50,17,18,11,6,6
7,1,44,50,20,18,9,6,5
8,2,42,50,18,17,11,6,5
13,2,43,50,17,18,9,6,5
15,2,44,50,17,16,9,6,5
16,3,42,50,19,16,9,6,5
21,3,43,50,17,16,9,7,5
23,3,44,50,18,16,10,6,5
24,4,42,50,17,17,10,6,5


In [10]:
# Stage H: prevalence tables and the provenance report.
prev_rows = []
for f in range(1, N_FOLDS + 1):
    for split_name, frame in [
        ("test", cohort[cohort["test_fold"] == f]),
        ("fit", cohort[[roles.get((f, i)) == "fit" for i in cohort["image_id"]]]),
        ("inner_val", cohort[[roles.get((f, i)) == "inner_val" for i in cohort["image_id"]]]),
    ]:
        prev_rows.append({"fold": f, "split": split_name, "images": len(frame),
                          **{t: int(frame[t].sum()) for t in TARGETS},
                          **{f"src_{s}": int((frame["source_dataset"] == s).sum()) for s in SOURCES}})
prevalence = pd.DataFrame(prev_rows)
prevalence.to_csv(PHASE0_REPORTS / "split_prevalence.csv", index=False)

artifact_paths = sorted([
    PHASE0_ARTIFACTS / "cohort.csv",
    PHASE0_ARTIFACTS / "outer_folds.csv",
    PHASE0_ARTIFACTS / "inner_splits.csv",
    PHASE0_ARTIFACTS / "cache_manifest.csv",
] + list(SUBSETS_DIR.glob("subset_*.csv")))
hashes = {str(p.relative_to(PROJECT_ROOT)): sha256_file(p) for p in artifact_paths}
provenance = {
    "environment": ENV,
    "constants": {
        "targets": TARGETS, "split_seed": SPLIT_SEED,
        "outer_test_capacity": OUTER_TEST_CAPACITY, "fit_images_per_fold": FIT_IMAGES_PER_FOLD,
        "subset_sizes": SUBSET_SIZES, "seed_subset_sizes": {str(k): v for k, v in SEED_SUBSET_SIZES.items()},
        "preprocessing_version": PREPROCESSING_VERSION,
        "label_table_sha256": LABEL_TABLE_SHA256,
        "phash_hamming_max": PHASH_HAMMING_MAX,
        "hires_verifier": {
            "side": HIRES_SIDE, "orb_features": HIRES_ORB_FEATURES,
            "fast_threshold": HIRES_FAST_THRESHOLD, "clahe_clip": HIRES_CLAHE_CLIP,
            "clahe_grid": HIRES_CLAHE_GRID, "mask_luminance": HIRES_MASK_LUMINANCE,
            "mask_erode": HIRES_MASK_ERODE, "ransac_px": HIRES_RANSAC_PX,
            "ransac_iters": HIRES_RANSAC_ITERS, "ransac_confidence": HIRES_RANSAC_CONFIDENCE,
            "rng_seed": HIRES_RNG_SEED, "confirm_inliers": HIRES_CONFIRM_INLIERS,
            "review_inliers": HIRES_REVIEW_INLIERS,
        },
    },
    "cohort_sha256": COHORT_SHA,
    "confirmed_duplicate_pairs": [list(p) for p in confirmed_pairs],
    "artifact_sha256": hashes,
}
write_json(PHASE0_ARTIFACTS / "provenance.json", provenance)
print(f"provenance.json written with {len(hashes)} artifact hashes")
prevalence.head(15)


provenance.json written with 44 artifact hashes


,fold,split,images,DR,NORMAL,ODC,MH,DN,src_aria,src_rfmid,src_stare
0,1,test,298,99,95,53,34,32,22,246,30
1,1,fit,1044,347,354,184,118,114,76,863,105
2,1,inner_val,147,49,44,26,17,16,11,121,15
3,2,test,298,99,98,52,34,32,22,246,30
4,2,fit,1044,347,348,185,118,114,76,863,105
5,2,inner_val,147,49,47,26,17,16,11,121,15
6,3,test,298,99,103,53,34,33,22,246,30
7,3,fit,1044,347,341,184,118,113,76,863,105
8,3,inner_val,147,49,49,26,17,16,11,121,15
9,4,test,298,99,98,53,34,33,22,246,30


In [11]:
# Finish checks. Each check re-reads the artifact files from disk.
checks = []

c = pd.read_csv(PHASE0_ARTIFACTS / "cohort.csv", dtype={"image_id": str})
checks.append(("cohort holds 1,489 unique images",
               len(c) == EXPECTED_COHORT and c["image_id"].nunique() == EXPECTED_COHORT))

audit = pd.read_csv(PHASE0_REPORTS / "crop_audit.csv", dtype={"image_id": str})
checks.append(("every image passes the crop audit",
               len(audit) == EXPECTED_COHORT and bool(audit["ok"].all())))

folds = pd.read_csv(PHASE0_ARTIFACTS / "outer_folds.csv", dtype={"image_id": str}).merge(
    c[["image_id", "analysis_group_id"] + TARGETS], on="image_id")
group_folds = folds.groupby("analysis_group_id")["test_fold"].nunique()
checks.append(("no analysis group appears in two outer folds", bool((group_folds == 1).all())))

inner = pd.read_csv(PHASE0_ARTIFACTS / "inner_splits.csv", dtype={"image_id": str}).merge(
    c[["image_id"] + TARGETS], on="image_id")
support_ok = True
for f in range(1, N_FOLDS + 1):
    for name, frame in [("test", folds[folds["test_fold"] == f]),
                        ("inner_val", inner[(inner["test_fold"] == f) & (inner["role"] == "inner_val")])]:
        for t in TARGETS:
            pos = int(frame[t].sum())
            if pos == 0 or pos == len(frame):
                support_ok = False
checks.append(("every evaluation set contains both values for each target", support_ok))

nesting_ok = True
for f in range(1, N_FOLDS + 1):
    for seed, sizes in SEED_SUBSET_SIZES.items():
        chain = [set(pd.read_csv(PHASE0_ARTIFACTS / "subsets" / f"subset_f{f}_s{seed}_n{n}.csv",
                                 dtype={"image_id": str})["image_id"])
                 for n in sorted(sizes)]
        for small, large in zip(chain, chain[1:]):
            if not small.issubset(large):
                nesting_ok = False
checks.append(("every smaller subset is part of the next larger subset", nesting_ok))

result = pd.DataFrame(checks, columns=["check", "pass"])
print(result.to_string(index=False))
if not result["pass"].all():
    raise PhaseStop("a finish check failed. Phase 0 is not complete.")
print("\nPHASE 0 COMPLETE")


                                                    check  pass
                         cohort holds 1,489 unique images  True
                        every image passes the crop audit  True
             no analysis group appears in two outer folds  True
every evaluation set contains both values for each target  True
   every smaller subset is part of the next larger subset  True

PHASE 0 COMPLETE
